# NDVI Temporal Analysis - Limarí Valley, Chile
## Drought monitoring using Google Earth Engine and Sentinel-2
### Author: Constanza Morales | 2026

## 1. Introduction

This notebook analyzes vegetation changes in the Limarí Valley, Coquimbo Region, Chile,
using Sentinel-2 satellite imagery and Google Earth Engine. The Limarí Valley has been 
severely affected by the megadrought that has impacted central Chile since 2010, making 
it an ideal study area for NDVI-based drought monitoring.

**Study area:** Limarí Valley, Coquimbo Region, Chile  
**Period:** 2016 - 2024  
**Data source:** Sentinel-2 (ESA) via Google Earth Engine  
**Tools:** Python, Google Earth Engine, Geemap

In [18]:
# Libraries
import ee
import geemap
import matplotlib.pyplot as plt

# Initialize Google Earth Engine
ee.Initialize(project='earth-engine-portfolio-498609')
print("GEE connected successfully")

GEE connected successfully


## 2. Study Area Definition

Defining the region of interest (ROI) over the Limarí Valley, Coquimbo Region, Chile.
The valley is located between 30°S and 31°S latitude, characterized by a semi-arid 
climate and highly dependent on irrigation agriculture.

In [19]:
# Load administrative boundaries
chile_admin = ee.FeatureCollection('FAO/GAUL/2015/level2')

# Filter Limarí province
limari = chile_admin.filter(ee.Filter.eq('ADM2_NAME', 'Limari'))

# Use Limarí province as ROI
roi = limari.geometry()

# Display map centered on study area
Map = geemap.Map()
Map.centerObject(roi, zoom=9)
Map.addLayer(limari, {'color': 'lightgray'}, 'Limarí Province')
Map

Map(center=[-30.789071339881666, -70.95653580542321], controls=(WidgetControl(options=['position', 'transparen…

## 3. Sentinel-2 Data Loading

Loading Sentinel-2 imagery over the Limarí Valley for four years to analyze
vegetation changes over time: 2016, 2019, 2022 and 2024.

In [20]:
# Load Sentinel-2 Surface Reflectance collection
# Filter by location, date and cloud cover for four years

s2_2016 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2016-01-01', '2016-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .median())

s2_2019 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2019-01-01', '2019-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .median())

s2_2022 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2022-01-01', '2022-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .median())

s2_2024 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2024-01-01', '2024-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .median())

print("Images loaded for 2016, 2019, 2022 and 2024")

Images loaded for 2016, 2019, 2022 and 2024


In [29]:
# Visualization parameters for true color (RGB)
vis_params = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000,
    'gamma': 1.4
}

# Add all four years to the map
Map2 = geemap.Map()
Map2.centerObject(roi, zoom=9)
Map2.addLayer(s2_2016, vis_params, 'Sentinel-2 2016')
Map2.addLayer(s2_2019, vis_params, 'Sentinel-2 2019')
Map2.addLayer(s2_2022, vis_params, 'Sentinel-2 2022')
Map2.addLayer(s2_2024, vis_params, 'Sentinel-2 2024')
Map2.addLayer(limari, {'color': 'lightgray', 'fillColor': '00000000'}, 'Limarí Province')
Map2

Map(center=[-30.78907133988169, -70.9565358054232], controls=(WidgetControl(options=['position', 'transparent_…

## 4. Sentinel-2 Band Structure

Exploring the band composition of Sentinel-2 imagery to understand 
which bands we will use for NDVI calculation.

In [22]:
# Explore band names of Sentinel-2 image
print("Sentinel-2 bands:")
print(s2_dry.bandNames().getInfo())

Sentinel-2 bands:
['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE']


### Key bands for this analysis:

| Band | Name | Wavelength | Use |
|------|------|------------|-----|
| B2 | Blue | 490 nm | RGB visualization |
| B3 | Green | 560 nm | RGB visualization |
| B4 | Red | 665 nm | RGB visualization / NDVI |
| B8 | NIR | 842 nm | NDVI calculation |
| B11 | SWIR-1 | 1610 nm | Drought / moisture index |
| B12 | SWIR-2 | 2190 nm | Drought / moisture index |

**NDVI formula:** (B8 - B4) / (B8 + B4)

## 5. NDVI Calculation - Temporal Analysis

Calculating NDVI for four years to analyze vegetation changes over time.
NDVI (Normalized Difference Vegetation Index) = (B8 - B4) / (B8 + B4)
Values range from -1 to 1, where higher values indicate healthier vegetation.

In [23]:
# Function to calculate NDVI
def calculate_ndvi(year):
    image = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(roi)
        .filterDate(f'{year}-01-01', f'{year}-12-31')
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .median())
    
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return ndvi

# Calculate NDVI for four years
ndvi_2016 = calculate_ndvi(2016)
ndvi_2019 = calculate_ndvi(2019)
ndvi_2022 = calculate_ndvi(2022)
ndvi_2024 = calculate_ndvi(2024)

print("NDVI calculated for 2016, 2019, 2022 and 2024")

NDVI calculated for 2016, 2019, 2022 and 2024


In [30]:
# NDVI visualization parameters
ndvi_params = {
    'min': -0.2,
    'max': 0.6,
    'palette': ['red', 'orange', 'yellow', 'lightgreen', 'green']
}

# Add all NDVI layers to map
Map3 = geemap.Map()
Map3.centerObject(roi, zoom=9)
Map3.addLayer(ndvi_2016, ndvi_params, 'NDVI 2016')
Map3.addLayer(ndvi_2019, ndvi_params, 'NDVI 2019')
Map3.addLayer(ndvi_2022, ndvi_params, 'NDVI 2022')
Map3.addLayer(ndvi_2024, ndvi_params, 'NDVI 2024')
Map3.addLayer(limari, {'color': 'lightgray', 'fillColor': '00000000'}, 'Limarí Province')
Map3

Map(center=[-30.78907133988169, -70.9565358054232], controls=(WidgetControl(options=['position', 'transparent_…

### NDVI Color Scale Interpretation

| Color | NDVI Range | Meaning |
|-------|-----------|---------|
| Red | -0.2 to 0.1 | No vegetation (bare soil, urban areas, water) |
| Orange | 0.1 to 0.2 | Very sparse vegetation |
| Yellow | 0.2 to 0.3 | Sparse vegetation |
| Light green | 0.3 to 0.45 | Moderate vegetation |
| Green | 0.45 to 0.6 | Dense healthy vegetation |

**Key observation:** Between 2016 and 2022 a significant reduction in vegetation 
cover is visible, consistent with the megadrought affecting central Chile. 
A partial recovery is observed in 2024.

## 6. NDVI Difference Analysis

Two difference maps are calculated to tell a complete story of the megadrought impact and recovery:

1. **2022 - 2016:** Quantifies vegetation loss during the megadrought peak
2. **2024 - 2022:** Quantifies vegetation recovery in recent years

Negative values indicate vegetation loss. Positive values indicate recovery.

In [31]:
# Calculate NDVI difference 1: Impact of megadrought (2022 - 2016)
ndvi_diff_drought = ndvi_2022.subtract(ndvi_2016).rename('NDVI_drought_impact')

# Calculate NDVI difference 2: Recent recovery (2024 - 2022)
ndvi_diff_recovery = ndvi_2024.subtract(ndvi_2022).rename('NDVI_recovery')

# Visualization parameters for difference maps
diff_params = {
    'min': -0.3,
    'max': 0.3,
    'palette': ['red', 'white', 'green']
}

# Display both difference maps
Map4 = geemap.Map()
Map4.centerObject(roi, zoom=9)
Map4.addLayer(ndvi_diff_drought, diff_params, 'Drought Impact 2022-2016')
Map4.addLayer(ndvi_diff_recovery, diff_params, 'Recovery 2024-2022')
Map4.addLayer(limari, {'color': 'lightgray', 'fillColor': '00000000'}, 'Limarí Province')
Map4

Map(center=[-30.789071339881666, -70.95653580542321], controls=(WidgetControl(options=['position', 'transparen…

### Difference Maps Interpretation

**Drought Impact (2022 - 2016):**
The map shows widespread vegetation loss (red) across the Limarí Valley,
confirming the severe impact of the Chilean megadrought between 2016 and 2022.

**Recovery (2024 - 2022):**
Partial vegetation recovery is observed in 2024, particularly in areas outside
the valley floor. However, the core agricultural areas still show negative 
differences, suggesting incomplete recovery from the drought.

## 7. Interactive Map

Final interactive map combining all analysis layers: NDVI for each year
and both difference maps, allowing visual comparison of vegetation changes
across the study period.

In [32]:
# Create final interactive map with all layers
Map_final = geemap.Map()
Map_final.centerObject(roi, zoom=9)

# Add NDVI layers for each year
Map_final.addLayer(ndvi_2016, ndvi_params, 'NDVI 2016')
Map_final.addLayer(ndvi_2019, ndvi_params, 'NDVI 2019')
Map_final.addLayer(ndvi_2022, ndvi_params, 'NDVI 2022')
Map_final.addLayer(ndvi_2024, ndvi_params, 'NDVI 2024')

# Add difference maps
Map_final.addLayer(ndvi_diff_drought, diff_params, 'Drought Impact 2022-2016')
Map_final.addLayer(ndvi_diff_recovery, diff_params, 'Recovery 2024-2022')

# Add rivers
rivers = ee.FeatureCollection('WWF/HydroSHEDS/v1/FreeFlowingRivers')
rivers_roi = rivers.filterBounds(roi)
Map_final.addLayer(rivers_roi, {'color': '4169E1'}, 'Rivers')

# Add Limarí Province boundary
Map_final.addLayer(limari, {'color': 'lightgray', 'fillColor': '00000000'}, 'Limarí Province')

# Add labels on top
Map_final.add_basemap('CartoDB.PositronOnlyLabels')

Map_final

Map(center=[-30.789071339881666, -70.95653580542321], controls=(WidgetControl(options=['position', 'transparen…

## 8. Conclusions

- The Limarí Valley experienced significant vegetation loss between 2016 and 2022, 
  consistent with the Chilean megadrought that has affected central Chile since 2010.
- NDVI values show a progressive decline from 2016 to 2022, with the most severe 
  vegetation loss concentrated in the valley floor and agricultural areas.
- A partial vegetation recovery is observed in 2024, likely associated with above-average 
  precipitation events in 2023-2024.
- The combination of Sentinel-2 imagery and Google Earth Engine allows efficient 
  monitoring of drought impacts over large areas at no cost.

**Data source:** Sentinel-2 SR Harmonized (ESA) via Google Earth Engine  
**Methods:** NDVI temporal analysis, difference mapping  
**Tools:** Python, Google Earth Engine, Geemap

In [33]:
# Export interactive map as HTML
Map_final.to_html('ndvi_limari_map.html')
print("Map exported successfully: ndvi_limari_map.html")

Map exported successfully: ndvi_limari_map.html
